# WhisperWatch live demo

Select **Runtime → Change runtime type → T4 GPU**, then run the cells in order. Have the team’s verified classifier ZIP ready for the upload prompt. Upload an English clip up to 30 seconds. The first transcription downloads about 3.1 GB of Whisper weights.

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys
from tempfile import TemporaryDirectory

from google.colab import files

if not shutil.which("nvidia-smi") or subprocess.run(["nvidia-smi", "-L"], capture_output=True).returncode:
    raise RuntimeError("Select a T4 GPU runtime before continuing.")

project = Path("/content/Whisperwatch")
if not (project / "app.py").is_file():
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/idotpixel/Whisperwatch.git", str(project)],
        check=True,
    )
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-live.txt",
     "nvidia-cublas-cu12", "nvidia-cudnn-cu12==9.*"],
    cwd=project, check=True,
)
uploads = files.upload()
if len(uploads) != 1:
    raise ValueError("Upload one approved classifier ZIP.")
with TemporaryDirectory() as folder:
    bundle = Path(folder) / "model.zip"
    bundle.write_bytes(next(iter(uploads.values())))
    subprocess.run([sys.executable, "-m", "src.setup_demo", str(bundle)], cwd=project, check=True)
subprocess.run([sys.executable, "-m", "src.backend_smoke"], cwd=project, check=True)


## Open the website

Use the public Gradio link printed by the next cell. Keep the Colab runtime connected while you use or share the app.

In [ ]:
sys.path.insert(0, str(project))
from src.colab_backend import gpu_process_env

subprocess.run(
    [sys.executable, "-u", "app.py", "--share"],
    cwd=project, env=gpu_process_env(), check=True,
)
